# LifeSnaps Deep Learning Pipeline Summary From Scripts

This notebook summarizes the reproducible `scripts/` pipeline and its outputs for the LifeSnaps sleep prediction project.

Current project framing:

- This is a deep learning project.
- Logistic Regression, Random Forest, DummyClassifier, and related tuning outputs are traditional ML baseline/reference material only.
- PCA is exploratory feature-structure analysis, not the default deep learning input.
- The main deep learning path starts from participant-date daily data before PCA and continues through rolling sequence tensors.

Current completed flow:

```text
MongoDB/raw source orientation
-> collection EDA and variable extraction
-> participant-date daily aggregation
-> merged daily modeling dataset
-> missing-value handling and feature finalization
-> exploratory PCA
-> participant-aware split
-> traditional ML reference baselines
-> deep learning sequence dataset creation
```

Current unfinished deep learning work:

```text
MLP baseline
SimpleRNN
LSTM
GRU
BiLSTM
1D CNN
model comparison against traditional ML reference baseline
```


## 0. Notebook Scope

이 노트북은 무거운 MongoDB 원본 처리를 다시 실행하기 위한 파일이 아니라, 지금까지 스크립트로 진행한 내용을 확인하고 발표/보고서용으로 정리하기 위한 파일입니다.

- 원본 MongoDB 접근과 대용량 집계는 `scripts/`에서 재현합니다.
- 이 노트북에서는 생성된 CSV와 report를 읽어 진행 상태를 검토합니다.
- `data/raw/` 파일은 수정하지 않습니다.
- `data/processed/` 결과를 기준으로 shape, key 중복, target 분포, 병합 coverage를 확인합니다.


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
REPORTS_DIR = PROJECT_ROOT / "reports"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
EXTRACTED_DIR = RAW_DIR / "extracted_variables"
DAILY_DIR = PROJECT_ROOT / "data" / "processed" / "daily_aggregates"
MODEL_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_dataset_daily.csv"

PROJECT_ROOT


## 1. Script Pipeline Map

아래 표는 현재까지의 진행 단계와 담당 스크립트입니다.


In [ ]:
script_map = pd.DataFrame(
    [
        {
            "step": "MongoDB raw overview",
            "script": "01_load_mongodb_raw.py",
            "main_output": "reports/mongodb_raw_overview.md",
            "status": "done",
        },
        {
            "step": "Fitbit type profiling",
            "script": "02_profile_fitbit_types.py",
            "main_output": "reports/fitbit_type_profile.md",
            "status": "done",
        },
        {
            "step": "Sleep raw extraction",
            "script": "03_export_sleep_from_mongodb.py",
            "main_output": "data/raw/sleep_from_mongodb.csv",
            "status": "done",
        },
        {
            "step": "Collection-level EDA",
            "script": "04_profile_collections_for_eda.py",
            "main_output": "reports/collection_eda_summary.md",
            "status": "done",
        },
        {
            "step": "Needed variable extraction",
            "script": "05_extract_needed_variables.py",
            "main_output": "data/raw/extracted_variables/*.csv",
            "status": "done",
        },
        {
            "step": "Date-level aggregation",
            "script": "06_aggregate_daily_features.py",
            "main_output": "data/processed/daily_aggregates/*.csv",
            "status": "done",
        },
        {
            "step": "Daily aggregate validation and visualization",
            "script": "07_validate_visualize_daily_aggregates.py",
            "main_output": "reports/daily_aggregation_validation.md",
            "status": "done",
        },
        {
            "step": "Merge modeling dataset",
            "script": "08_merge_daily_modeling_dataset.py",
            "main_output": "data/processed/modeling_dataset_daily.csv",
            "status": "done",
        },
        {
            "step": "Final dataset EDA",
            "script": "09_final_dataset_eda.py",
            "main_output": "reports/final_dataset_eda.md",
            "status": "done",
        },
        {
            "step": "Missing-value handling",
            "script": "10_handle_missing_values.py",
            "main_output": "data/processed/modeling_dataset_missing_handled.csv",
            "status": "done",
        },
        {
            "step": "Categorical encoding / feature finalization",
            "script": "11_encode_categorical_features.py",
            "main_output": "data/processed/modeling_dataset_encoded.csv",
            "status": "done",
        },
        {
            "step": "Exploratory feature scaling",
            "script": "12_scale_features.py",
            "main_output": "data/processed/modeling_dataset_scaled.csv",
            "status": "done",
        },
        {
            "step": "Exploratory PCA feature-structure analysis",
            "script": "13_run_pca.py",
            "main_output": "data/processed/modeling_dataset_pca.csv",
            "status": "done",
        },
        {
            "step": "Participant-aware train/test split",
            "script": "14_create_participant_split.py",
            "main_output": "data/processed/splits/*.csv",
            "status": "done",
        },
        {
            "step": "Traditional ML baseline reference",
            "script": "15_train_baseline_models.py",
            "main_output": "reports/baseline_modeling_summary.md",
            "status": "done",
        },
        {
            "step": "Traditional ML feature-set reference",
            "script": "16_compare_feature_sets.py",
            "main_output": "reports/feature_set_comparison_summary.md",
            "status": "done",
        },
        {
            "step": "Traditional ML baseline tuning reference",
            "script": "17_tune_baseline_models.py",
            "main_output": "reports/tuned_modeling_summary.md",
            "status": "done",
        },
        {
            "step": "Traditional ML baseline validation reference",
            "script": "18_validate_interpret_baseline_model.py",
            "main_output": "reports/final_baseline_validation_report.md",
            "status": "done",
        },
        {
            "step": "Deep learning sequence dataset creation",
            "script": "19_create_deep_learning_sequences.py",
            "main_output": "data/processed/deep_learning_sequences/*",
            "status": "done",
        },
        {
            "step": "Deep learning project roadmap",
            "script": "-",
            "main_output": "reports/deep_learning_project_roadmap.md",
            "status": "done",
        },
        {
            "step": "Deep learning data loaders",
            "script": "20_prepare_deep_learning_loaders.py",
            "main_output": "planned",
            "status": "planned",
        },
        {
            "step": "MLP deep learning baselines",
            "script": "21_train_mlp_baselines.py",
            "main_output": "planned",
            "status": "planned",
        },
        {
            "step": "SimpleRNN / LSTM / GRU / BiLSTM / 1D CNN sequence models",
            "script": "22_train_sequence_models.py",
            "main_output": "planned",
            "status": "planned",
        },
    ]
)

script_map


## 2. 컬렉션별 EDA 요약

`scripts/04_profile_collections_for_eda.py`에서 `fitbit`, `sema`, `surveys` 컬렉션을 확인했습니다.

핵심 내용:

- `fitbit`: sleep, stress, HRV, activity, SpO2, respiration, wrist temperature 후보 type 확인
- `sema`: Context and Mood Survey 중심의 일중 감정/맥락 응답 확인
- `surveys`: stai, panas, breq 등 참가자 단위 설문 type 확인


In [ ]:
collection_report = REPORTS_DIR / "collection_eda_summary.md"
print(collection_report)
print(collection_report.read_text(encoding="utf-8")[:2500])


## 3. 필요한 변수 추출 요약

`scripts/05_extract_needed_variables.py`에서 모델 후보 변수들을 raw extract table로 저장했습니다.

- 저장 위치: `data/raw/extracted_variables/`
- 이 단계는 병합/결측치 처리/스케일링/PCA를 하지 않습니다.
- `steps`, `calories`, `Wrist Temperature`처럼 큰 원본은 이후 날짜 집계 단계에서 MongoDB에서 직접 집계했습니다.


In [ ]:
extracted_shapes = []
for path in sorted(EXTRACTED_DIR.glob("*.csv")):
    header = pd.read_csv(path, nrows=0)
    with path.open("r", encoding="utf-8-sig") as file:
        rows = max(sum(1 for _ in file) - 1, 0)
    extracted_shapes.append({"file": path.name, "rows": rows, "columns": len(header.columns)})

pd.DataFrame(extracted_shapes)


## 4. 날짜 단위 집계 요약

`scripts/06_aggregate_daily_features.py`에서 집계 단위를 다음으로 정했습니다.

```text
participant_object_id + calendar_date
```

선정 이유:

- sleep target이 참가자별 하루 수면 결과입니다.
- Fitbit stress/HRV/activity/SpO2/호흡/체온은 같은 날짜 축으로 정렬할 수 있습니다.
- SEMA는 하루 여러 번 응답되므로 일별 count/rate로 요약했습니다.
- surveys는 반복 일별 데이터가 아니므로 participant-level summary로 유지했습니다.


In [ ]:
daily_shapes = []
for path in sorted(DAILY_DIR.glob("*.csv")):
    header = pd.read_csv(path, nrows=0)
    with path.open("r", encoding="utf-8-sig") as file:
        rows = max(sum(1 for _ in file) - 1, 0)
    daily_shapes.append({"file": path.name, "rows": rows, "columns": len(header.columns)})

pd.DataFrame(daily_shapes)


## 5. 집계 데이터 검증 및 시각화 요약

`scripts/07_validate_visualize_daily_aggregates.py`에서 다음을 확인했습니다.

- 각 집계 테이블의 row/column 수
- `participant_object_id + calendar_date` 중복 여부
- 날짜 범위
- missing rate
- sleep target 분포
- 주요 feature 분포

검증 중 `fitbit_calories_daily.csv`의 날짜 key 중복 문제가 발견되어 `scripts/06_aggregate_daily_features.py`에서 수정했고, 최종적으로 모든 집계 테이블의 duplicate key는 0입니다.


In [ ]:
duplicate_checks = []
for path in sorted(DAILY_DIR.glob("*.csv")):
    df = pd.read_csv(path)
    key = ["participant_object_id"] if path.name == "surveys_participant_summary.csv" else ["participant_object_id", "calendar_date"]
    duplicate_checks.append(
        {
            "file": path.name,
            "key": " + ".join(key),
            "duplicate_key_rows": int(df.duplicated(key).sum()),
        }
    )

pd.DataFrame(duplicate_checks)


### Generated Figures

아래 그림들은 `reports/figures/`에 저장된 집계 검증용 시각화입니다.


![sleep target distribution](../reports/figures/sleep_target_distribution.png)

![sleep metric histograms](../reports/figures/sleep_metric_histograms.png)

![daily table row coverage](../reports/figures/daily_table_row_coverage.png)

![key feature boxplots](../reports/figures/key_feature_boxplots.png)

![sleep daily time series](../reports/figures/sleep_daily_time_series.png)


## 6. 세 데이터 병합 요약

`scripts/08_merge_daily_modeling_dataset.py`에서 sleep target table을 기준으로 feature table들을 left join했습니다.

- base: `sleep_daily_target.csv`
- daily feature join key: `participant_object_id + calendar_date`
- survey feature join key: `participant_object_id`
- output: `data/processed/modeling_dataset_daily.csv`

수면 예측이 목표이므로 sleep target이 있는 participant-day만 최종 행으로 유지했습니다.


In [ ]:
model_df = pd.read_csv(MODEL_DATASET_PATH)

summary = {
    "rows": len(model_df),
    "columns": len(model_df.columns),
    "participants": model_df["participant_object_id"].nunique(),
    "duplicate_participant_date_rows": int(model_df.duplicated(["participant_object_id", "calendar_date"]).sum()),
    "overall_missing_rate": model_df.isna().mean().mean(),
}

summary


In [ ]:
model_df["good_sleep_label"].value_counts(dropna=False).sort_index().rename("rows").to_frame()


In [ ]:
merge_report = REPORTS_DIR / "merge_summary.md"
print(merge_report)
print(merge_report.read_text(encoding="utf-8")[:3500])


## 7. Final Dataset EDA Summary

`scripts/09_final_dataset_eda.py`에서 병합된 최종 데이터셋을 확인했습니다.

- input: `data/processed/modeling_dataset_daily.csv`
- report: `reports/final_dataset_eda.md`
- figures: `reports/figures/final_dataset_eda/`

이 단계에서는 결측치 처리, 범주형 인코딩, 스케일링, PCA를 하지 않고 EDA만 수행했습니다.


In [ ]:
final_eda_report = REPORTS_DIR / "final_dataset_eda.md"
print(final_eda_report)
print(final_eda_report.read_text(encoding="utf-8")[:4500])


### Final Dataset EDA Figures

![target distribution](../reports/figures/final_dataset_eda/target_distribution.png)

![missing rate by feature family](../reports/figures/final_dataset_eda/missing_rate_by_feature_family.png)

![key features by target](../reports/figures/final_dataset_eda/key_features_by_target.png)

![top target correlations](../reports/figures/final_dataset_eda/top_target_correlations.png)


## 8. Missing-Value Handling Summary

`scripts/10_handle_missing_values.py`에서 결측치 처리를 진행했습니다.

- input: `data/processed/modeling_dataset_daily.csv`
- output: `data/processed/modeling_dataset_missing_handled.csv`
- metadata: `data/processed/missing_value_feature_metadata.csv`
- report: `reports/missing_value_handling_summary.md`

이 단계에서는 같은 밤의 수면 결과 컬럼을 feature에서 제외했습니다. 결측률 70% 초과 feature는 제거하고, 나머지 결측 feature에는 missing indicator를 추가한 뒤 count/rate/sum 계열은 0, 그 외 numeric feature는 median으로 채웠습니다.


In [ ]:
missing_handled_path = PROJECT_ROOT / "data" / "processed" / "modeling_dataset_missing_handled.csv"
missing_meta_path = PROJECT_ROOT / "data" / "processed" / "missing_value_feature_metadata.csv"

missing_df = pd.read_csv(missing_handled_path)
missing_meta = pd.read_csv(missing_meta_path)

{
    "rows": len(missing_df),
    "columns": len(missing_df.columns),
    "missing_cells": int(missing_df.isna().sum().sum()),
    "duplicate_participant_date_rows": int(missing_df.duplicated(["participant_object_id", "calendar_date"]).sum()),
    "metadata_rows": len(missing_meta),
}


In [ ]:
missing_report = REPORTS_DIR / "missing_value_handling_summary.md"
print(missing_report)
print(missing_report.read_text(encoding="utf-8")[:4500])


## 9. Categorical Encoding / Feature Finalization Summary

`scripts/11_encode_categorical_features.py`에서 범주형 인코딩 단계를 진행했습니다.

- input: `data/processed/modeling_dataset_missing_handled.csv`
- output: `data/processed/modeling_dataset_encoded.csv`
- feature list: `data/processed/encoded_feature_columns.csv`
- report: `reports/categorical_encoding_summary.md`

현재 결측치 처리 결과의 object 컬럼은 `participant_object_id`, `calendar_date`뿐입니다. 둘 다 모델 predictor가 아니라 split/diagnostic용 key이므로 one-hot encoding하지 않았습니다. 따라서 이번 단계는 feature table finalization에 가깝습니다.


In [ ]:
encoded_path = PROJECT_ROOT / "data" / "processed" / "modeling_dataset_encoded.csv"
encoded_features_path = PROJECT_ROOT / "data" / "processed" / "encoded_feature_columns.csv"

encoded_df = pd.read_csv(encoded_path)
encoded_features = pd.read_csv(encoded_features_path)

{
    "rows": len(encoded_df),
    "columns": len(encoded_df.columns),
    "missing_cells": int(encoded_df.isna().sum().sum()),
    "duplicate_participant_date_rows": int(encoded_df.duplicated(["participant_object_id", "calendar_date"]).sum()),
    "object_columns": encoded_df.select_dtypes(include="object").columns.tolist(),
    "feature_columns": len(encoded_features),
    "feature_source_types": encoded_features["source_type"].value_counts().to_dict(),
}


In [ ]:
encoding_report = REPORTS_DIR / "categorical_encoding_summary.md"
print(encoding_report)
print(encoding_report.read_text(encoding="utf-8")[:3500])


## 10. Scaling Summary

`scripts/12_scale_features.py`에서 numeric feature scaling을 진행했습니다.

- input: `data/processed/modeling_dataset_encoded.csv`
- output: `data/processed/modeling_dataset_scaled.csv`
- scaler: `data/processed/standard_scaler.joblib`
- scaled feature summary: `data/processed/scaled_feature_columns.csv`
- dropped zero-variance features: `data/processed/scaling_dropped_features.csv`
- report: `reports/scaling_summary.md`

`participant_object_id`, `calendar_date`, `good_sleep_label`은 보존하고, model feature에만 `StandardScaler`를 적용했습니다. PCA 전에 정보가 없는 zero-variance feature 10개는 제거했습니다.


In [ ]:
scaled_path = PROJECT_ROOT / "data" / "processed" / "modeling_dataset_scaled.csv"
scaled_features_path = PROJECT_ROOT / "data" / "processed" / "scaled_feature_columns.csv"
scaling_dropped_path = PROJECT_ROOT / "data" / "processed" / "scaling_dropped_features.csv"

scaled_df = pd.read_csv(scaled_path)
scaled_features = pd.read_csv(scaled_features_path)
scaling_dropped = pd.read_csv(scaling_dropped_path)
scaled_feature_cols = scaled_features["feature"].tolist()

{
    "rows": len(scaled_df),
    "columns": len(scaled_df.columns),
    "scaled_features": len(scaled_feature_cols),
    "dropped_zero_variance_features": len(scaling_dropped),
    "missing_cells": int(scaled_df.isna().sum().sum()),
    "duplicate_participant_date_rows": int(scaled_df.duplicated(["participant_object_id", "calendar_date"]).sum()),
    "max_abs_scaled_mean": float(scaled_df[scaled_feature_cols].mean().abs().max()),
    "max_scaled_std_deviation_from_1": float((scaled_df[scaled_feature_cols].std(ddof=0) - 1).abs().max()),
}


In [ ]:
scaling_report = REPORTS_DIR / "scaling_summary.md"
print(scaling_report)
print(scaling_report.read_text(encoding="utf-8")[:3500])


## 11. Exploratory PCA Summary

PCA was used only for exploratory feature-structure analysis. It is not the default input path for deep learning experiments.

Key points:

- Input: `data/processed/modeling_dataset_scaled.csv`
- Output: `data/processed/modeling_dataset_pca.csv`
- PCA preserves `participant_object_id`, `calendar_date`, and `good_sleep_label`.
- PCA helps inspect variance structure, redundancy, and feature-family clustering.
- The main deep learning path uses participant-date daily features before PCA.


In [ ]:
pca_path = PROJECT_ROOT / "data" / "processed" / "modeling_dataset_pca.csv"
pca_variance_path = PROJECT_ROOT / "data" / "processed" / "pca_explained_variance.csv"
pca_top_loadings_path = PROJECT_ROOT / "data" / "processed" / "pca_top_loadings.csv"

pca_df = pd.read_csv(pca_path)
pca_variance = pd.read_csv(pca_variance_path)
pca_top_loadings = pd.read_csv(pca_top_loadings_path)

selected_pc_count = len([column for column in pca_df.columns if column.startswith("PC")])
selected_cumulative_variance = pca_variance.loc[
    pca_variance["component_number"] == selected_pc_count,
    "cumulative_explained_variance_ratio",
].iloc[0]

pd.DataFrame(
    [
        {
            "rows": len(pca_df),
            "columns": len(pca_df.columns),
            "selected_pc_count": selected_pc_count,
            "selected_cumulative_variance": selected_cumulative_variance,
            "missing_cells": int(pca_df.isna().sum().sum()),
            "duplicate_participant_dates": int(pca_df.duplicated(["participant_object_id", "calendar_date"]).sum()),
        }
    ]
)


In [ ]:
pca_variance.head(20)[
    ["component", "explained_variance_ratio", "cumulative_explained_variance_ratio"]
]


In [ ]:
pca_top_loadings.head(40)


In [ ]:
pca_report = REPORTS_DIR / "pca_summary.md"
print(pca_report.read_text(encoding="utf-8")[:5000])


### PCA Figures

![PCA explained variance ratio](../reports/figures/pca/explained_variance_ratio.png)

![PCA cumulative explained variance](../reports/figures/pca/cumulative_explained_variance.png)

![PC1 vs PC2 by target](../reports/figures/pca/pc1_pc2_by_target.png)


## 12. Participant-Aware Split Summary

The split keeps all rows from the same participant in the same split. This prevents participant leakage across train and test data.

This split supports both traditional ML reference baselines and deep learning sequence dataset creation.


In [ ]:
train_split_path = PROJECT_ROOT / "data" / "processed" / "splits" / "train_participant_split.csv"
test_split_path = PROJECT_ROOT / "data" / "processed" / "splits" / "test_participant_split.csv"
participant_assignments_path = PROJECT_ROOT / "data" / "processed" / "splits" / "participant_split_assignments.csv"

train_split = pd.read_csv(train_split_path)
test_split = pd.read_csv(test_split_path)
participant_assignments = pd.read_csv(participant_assignments_path)

overlap = set(train_split["participant_object_id"].astype(str)) & set(test_split["participant_object_id"].astype(str))

pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_split),
            "participants": train_split["participant_object_id"].nunique(),
            "target_mean": train_split["good_sleep_label"].mean(),
            "duplicate_participant_dates": int(train_split.duplicated(["participant_object_id", "calendar_date"]).sum()),
            "participant_overlap": len(overlap),
        },
        {
            "split": "test",
            "rows": len(test_split),
            "participants": test_split["participant_object_id"].nunique(),
            "target_mean": test_split["good_sleep_label"].mean(),
            "duplicate_participant_dates": int(test_split.duplicated(["participant_object_id", "calendar_date"]).sum()),
            "participant_overlap": len(overlap),
        },
    ]
)


In [ ]:
participant_assignments.head(20)


In [ ]:
split_report = REPORTS_DIR / "participant_split_summary.md"
print(split_report.read_text(encoding="utf-8")[:5000])


## 13. Traditional ML Baseline Reference Summary

This section summarizes traditional ML baseline results. These are reference baselines only and should not be described as deep learning results.

Models include DummyClassifier, Logistic Regression, Logistic Regression + PCA, and Random Forest.


In [ ]:
baseline_metrics_path = PROJECT_ROOT / "data" / "processed" / "baseline_model_metrics.csv"
rf_importance_path = PROJECT_ROOT / "data" / "processed" / "baseline_random_forest_feature_importance.csv"

baseline_metrics = pd.read_csv(baseline_metrics_path)
rf_importance = pd.read_csv(rf_importance_path)

baseline_metrics


In [ ]:
baseline_metrics.sort_values(["balanced_accuracy", "roc_auc"], ascending=False).head(1)


In [ ]:
rf_importance.head(25)


In [ ]:
baseline_report = REPORTS_DIR / "baseline_modeling_summary.md"
print(baseline_report.read_text(encoding="utf-8")[:5000])


### Traditional ML Baseline Figures

![Baseline metric comparison](../reports/figures/baseline/metric_comparison.png)

![Baseline ROC curves](../reports/figures/baseline/roc_curves.png)

![Baseline confusion matrices](../reports/figures/baseline/confusion_matrices.png)


## 14. Traditional ML Feature-Set Reference Summary

This section checks how traditional ML baseline performance changes across feature-set variants.

The goal is to understand whether missingness, availability, wearable-only features, or context/survey features drive the reference baseline.


In [ ]:
feature_set_metrics_path = PROJECT_ROOT / "data" / "processed" / "feature_set_comparison_metrics.csv"
feature_set_definitions_path = PROJECT_ROOT / "data" / "processed" / "feature_set_definitions.csv"

feature_set_metrics = pd.read_csv(feature_set_metrics_path)
feature_set_definitions = pd.read_csv(feature_set_definitions_path)

feature_set_metrics


In [ ]:
feature_set_metrics.sort_values(["balanced_accuracy", "roc_auc"], ascending=False).head(5)


In [ ]:
feature_set_definitions.groupby("feature_set")["feature"].count().rename("feature_count").to_frame()


In [ ]:
feature_set_report = REPORTS_DIR / "feature_set_comparison_summary.md"
print(feature_set_report.read_text(encoding="utf-8")[:5000])


### Traditional ML Feature-Set Reference Figures

![Feature-set balanced accuracy](../reports/figures/feature_set_comparison/balanced_accuracy_by_feature_set.png)

![Feature-set ROC AUC](../reports/figures/feature_set_comparison/roc_auc_by_feature_set.png)


## 15. Traditional ML Baseline Tuning Reference

This section summarizes compact Logistic Regression and Random Forest tuning as traditional ML reference work.

These tuned results are not final deep learning model selection.


In [ ]:
tuned_metrics_path = PROJECT_ROOT / "data" / "processed" / "tuned_model_metrics.csv"
tuned_cv_results_path = PROJECT_ROOT / "data" / "processed" / "tuned_model_cv_results.csv"

tuned_metrics = pd.read_csv(tuned_metrics_path)
tuned_cv_results = pd.read_csv(tuned_cv_results_path)

tuned_metrics


In [ ]:
tuned_metrics.sort_values(["balanced_accuracy", "roc_auc"], ascending=False).head(1)


In [ ]:
tuned_cv_results[["candidate", "feature_set", "rank_test_score", "mean_test_score", "std_test_score", "params"]].sort_values(
    ["feature_set", "candidate", "rank_test_score"]
).head(20)


In [ ]:
tuned_report = REPORTS_DIR / "tuned_modeling_summary.md"
print(tuned_report.read_text(encoding="utf-8")[:5000])


### Traditional ML Baseline Tuning Figures

![Tuned test metrics](../reports/figures/tuned_modeling/tuned_test_metrics.png)

![CV vs test balanced accuracy](../reports/figures/tuned_modeling/cv_vs_test_balanced_accuracy.png)


## 16. Traditional ML Baseline Validation Reference

This section validates and interprets the selected traditional ML reference baseline.

The current reference baseline is `wearable_only + Logistic Regression`. It is used only as a comparison point for deep learning experiments.


In [ ]:
baseline_coefficients_path = PROJECT_ROOT / "data" / "processed" / "baseline_logistic_coefficients.csv"
baseline_calibration_path = PROJECT_ROOT / "data" / "processed" / "baseline_probability_calibration.csv"
baseline_bootstrap_path = PROJECT_ROOT / "data" / "processed" / "baseline_participant_bootstrap_metrics.csv"

baseline_coefficients = pd.read_csv(baseline_coefficients_path)
baseline_calibration = pd.read_csv(baseline_calibration_path)
baseline_bootstrap = pd.read_csv(baseline_bootstrap_path)

pd.DataFrame(
    [
        {
            "selected_baseline": "wearable_only + Logistic Regression",
            "feature_count": len(baseline_coefficients),
            "calibration_bins": len(baseline_calibration),
            "bootstrap_repeats": len(baseline_bootstrap),
        }
    ]
)


In [ ]:
pd.concat(
    [
        baseline_coefficients[baseline_coefficients["coefficient"] > 0].head(10),
        baseline_coefficients[baseline_coefficients["coefficient"] < 0].head(10),
    ]
)[["feature", "coefficient", "direction_for_good_sleep"]]


In [ ]:
baseline_calibration


In [ ]:
bootstrap_summary = []
for metric in ["balanced_accuracy", "roc_auc", "f1"]:
    values = baseline_bootstrap[metric].dropna()
    bootstrap_summary.append(
        {
            "metric": metric,
            "mean": values.mean(),
            "p025": values.quantile(0.025),
            "median": values.quantile(0.5),
            "p975": values.quantile(0.975),
            "valid_repeats": len(values),
        }
    )
pd.DataFrame(bootstrap_summary)


In [ ]:
final_baseline_report = REPORTS_DIR / "final_baseline_validation_report.md"
print(final_baseline_report.read_text(encoding="utf-8")[:7000])


### Traditional ML Baseline Validation Figures

![Top Logistic Regression coefficients](../reports/figures/final_baseline_validation/top_logistic_coefficients.png)

![Probability calibration](../reports/figures/final_baseline_validation/probability_calibration.png)

![Participant bootstrap intervals](../reports/figures/final_baseline_validation/bootstrap_metric_intervals.png)


## 17. Deep Learning Refocus

The project is treated as a deep learning project.

Still valid for deep learning:

- MongoDB/raw source inspection
- EDA and daily aggregation
- participant-date modeling dataset
- missing handling and feature finalization
- exploratory PCA
- traditional ML reference baselines
- participant-aware split

Main deep learning path:

- rolling sequence tensor creation
- MLP baseline experiments
- SimpleRNN, LSTM, GRU, BiLSTM, and 1D CNN experiments
- model comparison against the traditional ML reference baseline


In [ ]:
deep_learning_starting_points = {
    "preferred_daily_table_before_pca": [
        str(PROJECT_ROOT / "data" / "processed" / "modeling_dataset_missing_handled.csv"),
        str(PROJECT_ROOT / "data" / "processed" / "modeling_dataset_encoded.csv"),
    ],
    "traditional_ml_baseline_reference": "wearable_only + Logistic Regression",
    "next_script": str(SCRIPTS_DIR / "19_create_deep_learning_sequences.py"),
    "candidate_windows": [7, 14, 30],
    "candidate_deep_learning_models": ["MLP", "SimpleRNN", "LSTM", "GRU", "BiLSTM", "1D CNN"],
}

deep_learning_starting_points


## 18. Current Pipeline Status

Completed:

```text
raw source orientation
collection EDA
variable extraction
daily aggregation
merged participant-date dataset
missing handling
feature finalization
exploratory PCA
participant-aware split
traditional ML reference baseline
deep learning sequence tensor creation
```

Current next stage:

```text
manual deep learning model experiments
```


## 19. Deep Learning Sequence Dataset Creation Summary

`scripts/19_create_deep_learning_sequences.py` creates the first deep-learning-ready tensor datasets from participant-date daily rows before PCA.

Important framing:

- Logistic Regression and Random Forest remain traditional ML baseline/reference only.
- PCA remains exploratory feature-structure analysis and is not used as the default deep learning input.
- The input table is `data/processed/modeling_dataset_encoded.csv` plus `data/processed/encoded_feature_columns.csv`.
- Rows are sorted by `participant_object_id` and `calendar_date`.
- The existing participant-aware train/test split is reused, and validation participants are split from the original training participants.
- Feature scaling is fit only on deep-learning train rows before transforming validation/test rows.
- Contiguous rolling windows are created for 7, 14, and 30 days.

Saved arrays support the planned model families:

- `X_sequence`: samples x time_steps x features for SimpleRNN, LSTM, GRU, BiLSTM, and 1D CNN.
- `X_mlp_flattened`: samples x time_steps*features for a window-flattened MLP.
- `X_mlp_current_day`: samples x features for a daily tabular MLP baseline at the window endpoint.


In [ ]:
sequence_summary_path = PROJECT_ROOT / "data" / "processed" / "deep_learning_sequences" / "sequence_tensor_summary.csv"
sequence_split_path = PROJECT_ROOT / "data" / "processed" / "deep_learning_sequences" / "deep_learning_participant_split_assignments.csv"
sequence_report_path = REPORTS_DIR / "deep_learning_sequence_dataset_summary.md"

sequence_summary = pd.read_csv(sequence_summary_path)
sequence_splits = pd.read_csv(sequence_split_path)

sequence_summary[[
    "window",
    "split",
    "samples",
    "participants",
    "sequence_shape",
    "mlp_flattened_shape",
    "mlp_current_day_shape",
    "class_0_samples",
    "class_1_samples",
    "good_sleep_label_mean",
]]


In [ ]:
sequence_splits["deep_learning_split"].value_counts().rename("participants").to_frame()


In [ ]:
print(sequence_report_path)
print(sequence_report_path.read_text(encoding="utf-8")[:6000])


## 20. Current Deep Learning Pipeline Status

Completed for deep learning:

```text
participant-date daily table
-> missing-value handling
-> categorical encoding / feature finalization
-> participant-aware train/test split
-> deep-learning train/validation/test participant split
-> train-only scaling for tensor creation
-> rolling sequence tensors for 7/14/30 day windows
```

Current tensor outputs:

```text
data/processed/deep_learning_sequences/window_7/train.npz
data/processed/deep_learning_sequences/window_7/validation.npz
data/processed/deep_learning_sequences/window_7/test.npz
data/processed/deep_learning_sequences/window_14/train.npz
data/processed/deep_learning_sequences/window_14/validation.npz
data/processed/deep_learning_sequences/window_14/test.npz
data/processed/deep_learning_sequences/window_30/train.npz
data/processed/deep_learning_sequences/window_30/validation.npz
data/processed/deep_learning_sequences/window_30/test.npz
```

Correct next work:

```text
Train initial MLP / SimpleRNN / LSTM / GRU / BiLSTM / 1D CNN experiments.
```


## 21. Deep Learning Project Roadmap

The full project roadmap is now tracked in `reports/deep_learning_project_roadmap.md`.

Current position:

```text
Deep learning data preparation is complete.
Deep learning model training has not started yet.
```

Traditional ML reports should be read only as reference baselines. PCA should be read only as exploratory feature-structure analysis.


In [ ]:
roadmap_path = REPORTS_DIR / "deep_learning_project_roadmap.md"
print(roadmap_path)
print(roadmap_path.read_text(encoding="utf-8")[:7000])


## 22. Pre-Modeling Design Report

`reports/pre_modeling_design_report.md` consolidates the design choices made before deep learning model training.

It documents:

- how EDA was performed,
- how preprocessing was organized,
- how missing values were handled,
- how PCA was used to inspect feature structure,
- why PCA is exploratory rather than the default deep learning input,
- and why the current deep learning tensors are ready for MLP / SimpleRNN / LSTM / GRU / BiLSTM / 1D CNN experiments.


In [ ]:
pre_modeling_design_report_path = REPORTS_DIR / "pre_modeling_design_report.md"
print(pre_modeling_design_report_path)
print(pre_modeling_design_report_path.read_text(encoding="utf-8")[:9000])


## 23. Deep Learning Model Comparison Roadmap

`reports/deep_learning_model_comparison_roadmap.md` defines the model comparison plan before training begins.

Included candidates:

- MLP current-day
- MLP flattened-window
- SimpleRNN
- LSTM
- GRU
- BiLSTM
- 1D CNN

The comparison plan is validation-first: use validation results for model/window/hyperparameter selection, evaluate the selected candidate once on held-out participant test data, and compare against `wearable_only + Logistic Regression` as a traditional ML reference baseline only.


In [ ]:
model_comparison_roadmap_path = REPORTS_DIR / "deep_learning_model_comparison_roadmap.md"
print(model_comparison_roadmap_path)
print(model_comparison_roadmap_path.read_text(encoding="utf-8")[:9000])


## 24. Manual Deep Learning Experiment Notebook

A separate runnable notebook was created for user-side deep learning verification:

```text
notebooks/03_deep_learning_model_experiments.ipynb
```

This keeps the pipeline summary notebook separate from long-running training outputs. The experiment notebook includes code for:

- tensor loading and participant-overlap checks,
- MLP current-day and MLP flattened-window training,
- SimpleRNN, LSTM, GRU, BiLSTM, and 1D CNN training,
- validation-first ranking,
- held-out test comparison against the traditional ML reference baseline,
- participant-level bootstrap skeleton for the selected candidate.

The notebook defaults to `RUN_FULL_EXPERIMENTS = False` so full training does not start accidentally.


## 25. Final Deep Learning Result Summary

### Purpose

This section records the current final deep learning result after same-date subset ablation, previous-day timing sensitivity analysis, and participant-level bootstrap uncertainty estimation.

### Completed Deep Learning Experiment Tracks

- **Phase 1A: same-date feature subset ablation**
  - Compared `full_current`, `no_high_sleep_session`, and `daytime_only` feature subsets.
  - Tested windows 7, 14, and 30.
  - Tested `mlp_current_day`, `GRU`, `BiLSTM`, and `1D CNN`.
  - Total: 36 experiments.

- **Phase 2A: previous-day timing sensitivity**
  - Used features from `target_date - 1 day` only.
  - Compared `daytime_only` and `no_high_sleep_session`.
  - Tested windows 7 and 14.
  - Tested `mlp_current_day`, `GRU`, `BiLSTM`, and `1D CNN`.
  - Total: 16 experiments.

- **Participant-level bootstrap uncertainty**
  - Estimated uncertainty by resampling test participants rather than rows.
  - Evaluated the final same-date candidate, sequence comparison candidate, window-14 sensitivity candidate, and best previous-day candidate.

### Current Best Candidate

| item | value |
| --- | --- |
| experiment_id | `phase1a_003` |
| feature timing | `same_date` |
| feature subset | `daytime_only` |
| window | `7` |
| model | `mlp_current_day` |
| validation-selected threshold | `0.42` |
| test balanced accuracy | `0.8440` |
| participant-bootstrap balanced accuracy 95% CI | `[0.8042, 0.8924]` |
| test ROC AUC | `0.9023` |
| participant-bootstrap ROC AUC 95% CI | `[0.8483, 0.9463]` |
| test F1 | `0.8215` |

### Sequence Model Comparison Candidate

| item | value |
| --- | --- |
| experiment_id | `phase1a_002` |
| feature timing | `same_date` |
| feature subset | `daytime_only` |
| window | `7` |
| model | `GRU` |
| test balanced accuracy | `0.8317` |
| participant-bootstrap balanced accuracy 95% CI | `[0.7886, 0.8859]` |

### Previous-Day Conservative Candidate

| item | value |
| --- | --- |
| experiment_id | `phase2a_006` |
| feature timing | `previous_day` |
| feature subset | `daytime_only` |
| window | `14` |
| model | `BiLSTM` |
| test balanced accuracy | `0.6098` |
| participant-bootstrap balanced accuracy 95% CI | `[0.4758, 0.7413]` |

### Interpretation

The strongest current model is a **same-date sleep classification model**, not a strict previous-day forecasting model. Earlier date-alignment diagnostics showed that the sleep target `calendar_date` matches the sleep end date. Therefore, same-date features can include information from the same calendar day as the sleep endpoint or information close to the target event.

The stricter previous-day experiment reduced temporal ambiguity by using only `target_date - 1 day` features, but performance dropped substantially. This indicates that strictly prior-day features are much weaker predictors in the current dataset.

### Final Reporting Guidance

Final claims should separate two tracks:

1. **Best same-date classifier**
   - `same_date / daytime_only / window 7 / mlp_current_day`
   - Strong test performance and stable participant-level bootstrap interval.

2. **Previous-day timing sensitivity analysis**
   - Previous-day features produce much lower and less certain performance.
   - Useful as a conservative control, but not the current best model.

### Key Output Artifacts

- `reports/deep_learning_subset_ablation_and_previous_day_report.md`
- `data/processed/deep_learning_previous_day/experiments/phase_2a_outputs/final_candidate_summary_for_report.csv`
- `data/processed/deep_learning_previous_day/experiments/phase_2a_outputs/final_korean_experiment_summary.md`
- `data/processed/deep_learning_previous_day/experiments/phase_2a_outputs/selected_candidate_participant_bootstrap_summary.csv`


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")
FINAL_SUMMARY_PATH = PROJECT_ROOT / "data" / "processed" / "deep_learning_previous_day" / "experiments" / "phase_2a_outputs" / "final_candidate_summary_for_report.csv"
BOOTSTRAP_PATH = PROJECT_ROOT / "data" / "processed" / "deep_learning_previous_day" / "experiments" / "phase_2a_outputs" / "selected_candidate_participant_bootstrap_summary.csv"
REPORT_PATH = PROJECT_ROOT / "reports" / "deep_learning_subset_ablation_and_previous_day_report.md"

print("final candidate summary exists:", FINAL_SUMMARY_PATH.exists())
print("bootstrap summary exists:", BOOTSTRAP_PATH.exists())
print("final report exists:", REPORT_PATH.exists())

final_candidate_summary = pd.read_csv(FINAL_SUMMARY_PATH, encoding="utf-8-sig")
bootstrap_summary = pd.read_csv(BOOTSTRAP_PATH, encoding="utf-8-sig")

display(final_candidate_summary)
display(bootstrap_summary[bootstrap_summary["metric"].isin(["balanced_accuracy", "roc_auc"])])


In [ ]:
# Final candidate compact view
# Purpose:
# - Keep the project-level final candidate table easy to review.
# - Do not rerun model training from this summary notebook.

summary_view = final_candidate_summary[
    [
        "role",
        "feature_timing",
        "experiment_id",
        "subset_name",
        "window",
        "model_family",
        "balanced_accuracy",
        "bal_acc_ci_lower",
        "bal_acc_ci_upper",
        "roc_auc",
        "roc_auc_ci_lower",
        "roc_auc_ci_upper",
        "f1",
        "precision",
        "recall",
        "interpretation",
    ]
].copy()

display(summary_view)

best = summary_view[summary_view["role"] == "best_same_date_classifier"].iloc[0]

print("Current best candidate")
print(f"- experiment_id: {best['experiment_id']}")
print(f"- model: {best['feature_timing']} / {best['subset_name']} / window {best['window']} / {best['model_family']}")
print(f"- balanced accuracy: {best['balanced_accuracy']:.4f}")
print(f"- balanced accuracy 95% CI: [{best['bal_acc_ci_lower']:.4f}, {best['bal_acc_ci_upper']:.4f}]")
print(f"- ROC AUC: {best['roc_auc']:.4f}")
print(f"- ROC AUC 95% CI: [{best['roc_auc_ci_lower']:.4f}, {best['roc_auc_ci_upper']:.4f}]")
print(f"- F1: {best['f1']:.4f}")


## 26. Phase 3A Previous-Day Rolling/Trend Follow-Up

### Purpose

This section records the follow-up strict previous-day forecasting experiment after the Phase 1A/2A final summary.

Phase 3A tested whether rolling and trend feature engineering could improve strict previous-day forecasting while keeping the conservative `daytime_only` feature family.

### Phase 3A Design

- Input dataset: `data/processed/deep_learning_previous_day/modeling_dataset_previous_day_encoded.csv`
- Base feature subset: `data/processed/deep_learning_feature_subsets/daytime_only_features.csv`
- Feature timing: strict previous-day, where `feature_date = target calendar_date - 1 day`
- Feature engineering:
  - 3-day and 7-day rolling mean
  - 3-day and 7-day rolling standard deviation
  - 3-day and 7-day rolling minimum
  - 3-day and 7-day rolling maximum
  - within-block 1-row difference
  - deviation from 3-day and 7-day rolling means
- Rolling history rule: participant-level contiguous blocks only; history does not cross date gaps.
- Final feature count after train-only zero-variance removal: `122`

### Phase 3A Experiments

| experiment_id | feature_timing | subset_name | model_family | window |
| --- | --- | --- | --- | ---: |
| `phase3a_000` | `previous_day` | `daytime_only_rolling_trend` | `mlp_current_day` | 1 |
| `phase3a_001` | `previous_day` | `daytime_only_rolling_trend` | `GRU` | 7 |
| `phase3a_002` | `previous_day` | `daytime_only_rolling_trend` | `GRU` | 14 |

### Main Result

The closest Phase 3A candidate was:

`phase3a_001 / previous_day / daytime_only_rolling_trend / window 7 / GRU`

It reached test balanced accuracy `0.6054`, close to but slightly below the current previous-day reference candidate `phase2a_006` at `0.6098`.

### Direct Previous-Day Comparison

| role | experiment_id | subset | model | window | threshold | test balanced accuracy | test ROC AUC | test F1 | test precision | test recall |
| --- | --- | --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| current previous-day reference | `phase2a_006` | `daytime_only` | `BiLSTM` | 14 | 0.51 | 0.6098 | 0.6063 | 0.5833 | 0.5895 | 0.5773 |
| close rolling/trend candidate | `phase3a_001` | `daytime_only_rolling_trend` | `GRU` | 7 | 0.35 | 0.6054 | 0.6385 | 0.6145 | 0.4880 | 0.8293 |

### Interpretation

Phase 3A did not justify replacing the current strict previous-day reference candidate. The rolling/trend GRU window-7 model had higher recall and F1 than `phase2a_006`, but lower precision and slightly lower balanced accuracy.

The strict previous-day reference remains:

`phase2a_006 / previous_day / daytime_only / window 14 / BiLSTM`

The overall best model remains:

`phase1a_003 / same_date / daytime_only / window 7 / mlp_current_day`

The overall best model must still be described as same-date sleep classification, not strict previous-day forecasting.

### Key Output Artifacts

- `reports/phase3a_previous_day_rolling_trend_report.md`
- `data/processed/deep_learning_previous_day/rolling_trend_daytime_only/tensor_summary.csv`
- `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3a_outputs/phase_3a_validation_ranked_test_summary.csv`
- `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3a_outputs/phase_3a_threshold_policy_comparison.csv`
- `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3a_outputs/phase2a_006_vs_phase3a_001_direct_comparison.csv`


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")
PHASE3A_REPORT_PATH = PROJECT_ROOT / "reports" / "phase3a_previous_day_rolling_trend_report.md"
PHASE3A_TENSOR_SUMMARY_PATH = PROJECT_ROOT / "data" / "processed" / "deep_learning_previous_day" / "rolling_trend_daytime_only" / "tensor_summary.csv"
PHASE3A_RANKING_PATH = PROJECT_ROOT / "data" / "processed" / "deep_learning_previous_day" / "experiments" / "phase_3a_rolling_trend_daytime_only" / "phase_3a_outputs" / "phase_3a_validation_ranked_test_summary.csv"
PHASE3A_DIRECT_COMPARISON_PATH = PROJECT_ROOT / "data" / "processed" / "deep_learning_previous_day" / "experiments" / "phase_3a_rolling_trend_daytime_only" / "phase_3a_outputs" / "phase2a_006_vs_phase3a_001_direct_comparison.csv"

print("Phase 3A report exists:", PHASE3A_REPORT_PATH.exists())
print("Phase 3A tensor summary exists:", PHASE3A_TENSOR_SUMMARY_PATH.exists())
print("Phase 3A ranking exists:", PHASE3A_RANKING_PATH.exists())
print("Phase 3A direct comparison exists:", PHASE3A_DIRECT_COMPARISON_PATH.exists())

phase3a_tensor_summary = pd.read_csv(PHASE3A_TENSOR_SUMMARY_PATH, encoding="utf-8-sig")
phase3a_ranking = pd.read_csv(PHASE3A_RANKING_PATH, encoding="utf-8-sig")
phase3a_direct_comparison = pd.read_csv(PHASE3A_DIRECT_COMPARISON_PATH, encoding="utf-8-sig")

print("\nPhase 3A tensor summary")
display(phase3a_tensor_summary)

print("\nPhase 3A validation-ranked paired test summary")
display(phase3a_ranking)

print("\nCorrected previous-day direct comparison")
display(phase3a_direct_comparison)


## 27. Phase 3B Seed Robustness Follow-Up

### Purpose

Phase 3B checked whether the closest Phase 3A rolling/trend GRU candidates were stable across random seeds.

The goal was not to introduce a new feature set, but to test whether the Phase 3A result was robust enough to challenge the existing strict previous-day reference candidate.

### Experiments

Phase 3B repeated two Phase 3A GRU candidates:

| base_experiment_id | feature_timing | subset_name | model_family | window | seeds |
| --- | --- | --- | --- | ---: | --- |
| `phase3a_001` | `previous_day` | `daytime_only_rolling_trend` | `GRU` | 7 | `7, 123, 2026` |
| `phase3a_002` | `previous_day` | `daytime_only_rolling_trend` | `GRU` | 14 | `7, 123, 2026` |

Training used the same validation balanced accuracy early-stopping and threshold-selection rule as Phase 3A.

### Seed Robustness Result

| base_experiment_id | model | window | runs | mean test balanced accuracy | std | min | max | mean ROC AUC | mean F1 | mean precision | mean recall | delta mean vs `phase2a_006` |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| `phase3a_001` | `GRU` | 7 | 3 | 0.5986 | 0.0231 | 0.5798 | 0.6244 | 0.6508 | 0.6074 | 0.4843 | 0.8157 | -0.0112 |
| `phase3a_002` | `GRU` | 14 | 3 | 0.5223 | 0.0468 | 0.4865 | 0.5752 | 0.5377 | 0.5763 | 0.4861 | 0.7113 | -0.0874 |

### Interpretation

`phase3a_001` remained the stronger rolling/trend candidate, but its mean seed-robust test balanced accuracy did not exceed the existing previous-day reference candidate `phase2a_006`.

One `phase3a_001` seed exceeded `phase2a_006`, but the average across seeds was lower. This suggests that the rolling/trend GRU window-7 result is close but not stable enough to replace the previous-day reference.

`phase3a_002` was weaker and less stable across seeds. Its strong original Phase 3A validation result appears not to transfer reliably to held-out test participants.

### Updated Conclusion

Phase 3B does not justify replacing the strict previous-day reference candidate.

Current strict previous-day reference:

`phase2a_006 / previous_day / daytime_only / window 14 / BiLSTM`

Close rolling/trend comparison candidate:

`phase3a_001 / previous_day / daytime_only_rolling_trend / window 7 / GRU`

Overall best model remains:

`phase1a_003 / same_date / daytime_only / window 7 / mlp_current_day`

The overall best model is still a same-date sleep classification model, not a strict previous-day forecasting model.

### Key Output Artifacts

- `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3b_seed_robustness_outputs/phase_3b_seed_robustness_model_metrics.csv`
- `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3b_seed_robustness_outputs/phase_3b_seed_robustness_training_history.csv`
- `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3b_seed_robustness_outputs/phase_3b_seed_robustness_model_predictions.csv`
- `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3b_seed_robustness_outputs/phase_3b_seed_robustness_summary.csv`
- `data/processed/deep_learning_previous_day/experiments/phase_3a_rolling_trend_daytime_only/phase_3b_seed_robustness_outputs/phase_3b_seed_robustness_summary.md`

In [ ]:
# Notebook UTF-8 / JSON / syntax validation
# 원하는 결과:
# - 02 pipeline summary notebook이 JSON으로 정상 로드되는지 확인한다.
# - 모든 code cell이 Python syntax상 유효한지 확인한다.
# - UTF-8 깨짐 흔적이 없는지 확인한다.
# - stored outputs 수를 확인한다.

import ast
import json
from pathlib import Path

PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "02_pipeline_from_scripts_summary.ipynb"

text = NOTEBOOK_PATH.read_text(encoding="utf-8")
nb = json.loads(text)

code_cell_count = 0
syntax_errors = []

for idx, cell in enumerate(nb["cells"]):
    if cell.get("cell_type") == "code":
        code_cell_count += 1
        source = "".join(cell.get("source", []))
        try:
            ast.parse(source, filename=f"cell_{idx}")
        except SyntaxError as exc:
            syntax_errors.append(
                {
                    "cell_index": idx,
                    "message": str(exc),
                    "line": exc.lineno,
                    "offset": exc.offset,
                }
            )

stored_output_count = sum(
    len(cell.get("outputs", []))
    for cell in nb["cells"]
    if cell.get("cell_type") == "code"
)

replacement_count = text.count("\ufffd")
mojibake_marker_count = (
    text.count("����")
    + text.count("???")
    + text.count("遺")
    + text.count("寃")
)

print("notebook:", NOTEBOOK_PATH)
print("cells:", len(nb["cells"]))
print("code cells:", code_cell_count)
print("syntax error count:", len(syntax_errors))
print("stored output count:", stored_output_count)
print("replacement char count:", replacement_count)
print("mojibake marker count:", mojibake_marker_count)

if syntax_errors:
    display(syntax_errors)
else:
    print("syntax validation passed")

if replacement_count == 0 and mojibake_marker_count == 0:
    print("UTF-8/mojibake check passed")
else:
    print("Check notebook text for possible encoding damage.")

notebook: c:\workSpace\DeepLearnin_sleep\notebooks\02_pipeline_from_scripts_summary.ipynb
cells: 87
code cells: 50
syntax error count: 0
stored output count: 0
replacement char count: 0
mojibake marker count: 0
syntax validation passed
UTF-8/mojibake check passed


## Phase 3A/3B Previous-Day Rolling/Trend Follow-Up

After the initial previous-day timing sensitivity analysis, an additional feature-engineering follow-up was run to test whether stricter previous-day forecasting could be improved.

### Phase 3A: Rolling/Trend Feature Engineering

Phase 3A used the previous-day dataset and started from the conservative `daytime_only` feature subset. Rolling and trend features were created within participant-level contiguous date blocks so that history did not cross date gaps.

Feature engineering included:

- 3-day and 7-day rolling mean
- 3-day and 7-day rolling standard deviation
- 3-day and 7-day rolling minimum
- 3-day and 7-day rolling maximum
- within-block 1-row difference
- deviation from 3-day and 7-day rolling means

After train-only zero-variance feature removal, the saved Phase 3A tensor set used 122 features.

The closest Phase 3A candidate was:

| experiment_id | feature timing | subset | model | window | test balanced accuracy | test ROC AUC | test F1 | test precision | test recall |
| --- | --- | --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |
| `phase3a_001` | previous-day | `daytime_only_rolling_trend` | GRU | 7 | 0.6054 | 0.6385 | 0.6145 | 0.4880 | 0.8293 |

### Phase 3B: Seed Robustness Check

Phase 3B repeated the closest rolling/trend GRU candidates across additional random seeds.

| base experiment | model | window | runs | mean test balanced accuracy | std | min | max | mean F1 | mean precision | mean recall |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| `phase3a_001` | GRU | 7 | 3 | 0.5986 | 0.0231 | 0.5798 | 0.6244 | 0.6074 | 0.4843 | 0.8157 |
| `phase3a_002` | GRU | 14 | 3 | 0.5223 | 0.0468 | 0.4865 | 0.5752 | 0.5763 | 0.4861 | 0.7113 |

### Updated Previous-Day Conclusion

The current previous-day reference candidate remains:

`phase2a_006 / previous_day / daytime_only / window 14 / BiLSTM`

The close rolling/trend comparison candidate is:

`phase3a_001 / previous_day / daytime_only_rolling_trend / window 7 / GRU`

`phase3a_001` had higher recall and F1 than `phase2a_006`, but lower precision and slightly lower balanced accuracy under validation-selected thresholding. The seed robustness check also showed that its mean test balanced accuracy did not exceed `phase2a_006`.

Therefore, Phase 3A/3B does not justify replacing the current strict previous-day reference candidate.

The overall best model remains:

`phase1a_003 / same_date / daytime_only / window 7 / mlp_current_day`

This overall best model should still be interpreted as same-date sleep classification, not strict previous-day forecasting.

### Strict Pre-Sleep Forecasting Final Status

The project objective was refined to strict pre-sleep forecasting: predicting the upcoming sleep episode's `good_sleep_label` using only data available before `sleep_start_datetime`.

Final selected model:
- Experiment family: `presleep_stage1_hp_tiny_dropout40_wd1e3`
- Representative checkpoint: `presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027`
- Feature set: Design C Stage 1 strict pre-sleep features
- Model: tiny regularized PyTorch MLP, hidden dims `(24, 12)`, dropout `0.40`, weight decay `0.001`
- Official threshold: `0.54`

Representative held-out test performance:
- Balanced accuracy: `0.6492`
- ROC AUC: `0.6937`
- Average precision: `0.6187`
- F1: `0.5153`
- Precision: `0.6553`
- Recall: `0.4245`

Seed robustness for best config:
- Mean test balanced accuracy: `0.6586`
- Std: `0.0078`
- Min/max: `0.6492 / 0.6677`

Participant bootstrap:
- Balanced accuracy 95% CI: `[0.5436, 0.7259]`
- ROC AUC 95% CI: `[0.5567, 0.7872]`

Calibration:
- Brier score: `0.2126`
- Expected calibration error: `0.1256`
- Probability scores should be treated cautiously; threshold classification/ranking is safer than direct probability interpretation.

Inference:
- Reusable feature builder and inference scripts were created under `src/pre_sleep_forecasting/`.
- End-to-end smoke test succeeded on 3 sample episodes.

## Strict Pre-Sleep Forecasting Final Result

The project objective was refined from same-date classification / previous-day forecasting to strict pre-sleep forecasting:

> Predict the upcoming sleep episode's `good_sleep_label` using only wearable-derived data available before `sleep_start_datetime`.

### Dataset

Final strict Design C Stage 1 dataset:

- Unit: sleep episode
- Rows: 3,551
- Participants: 69
- Prediction cutoff: `sleep_start_datetime`
- Split: participant-level train / validation / test
- Train / validation / test samples: 2,323 / 347 / 881
- Feature construction:
  - pre-sleep intraday steps
  - pre-sleep intraday calories
  - pre-sleep intraday heart rate
  - sleep-start timing/calendar features
  - previous-day daily activity/resting-HR features
  - missing indicators

Inference feature contract:

- Raw Stage 1 features: 70
- Zero-variance removed features: 12
- Final model input features: 58
- Preprocessing: raw 70 features -> train median imputer -> train StandardScaler -> remove zero-variance features -> 58 model features

### Model Search

All final pre-sleep models are PyTorch neural-network MLP models.

Tested model/feature variants:

| Candidate | Feature Set | Features | Test Balanced Accuracy | ROC AUC | Average Precision | F1 | Precision | Recall |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Stage 1 single seed | strict pre-sleep Stage 1 | 58 | 0.6338 | 0.6875 | 0.6009 | 0.4904 | 0.6275 | 0.4025 |
| Stage 1 seed mean | strict pre-sleep Stage 1 | 58 | 0.6107 | 0.6681 | 0.6016 | 0.5309 | 0.4942 | 0.6205 |
| Stage 2 full rolling | broad rolling/history | 380 | 0.6025 | 0.6628 | 0.5855 | 0.5307 | 0.4541 | 0.6384 |
| Stage 2B compact rolling | compact rolling/history | 148 | 0.5923 | 0.6852 | 0.5788 | 0.5530 | 0.4238 | 0.7956 |
| Best Stage 1 HP config mean | strict pre-sleep Stage 1 | 58 | 0.6586 | 0.6942 | 0.6185 | 0.5298 | 0.6736 | 0.4371 |

Rolling/history features increased recall but did not improve participant-level held-out balanced accuracy. The best generalization came from the simpler strict Stage 1 feature set with stronger regularization.

### Selected Model

Current best representative model:

- Experiment family: `presleep_stage1_hp_tiny_dropout40_wd1e3`
- Representative checkpoint: `presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027`
- Selection basis: validation balanced accuracy
- Feature set: strict pre-sleep Design C Stage 1
- Model: tiny regularized PyTorch MLP
- Hidden dimensions: `(24, 12)`
- Dropout: `0.40`
- Weight decay: `0.001`
- Official threshold: `0.54`

Representative held-out test performance:

- Balanced accuracy: `0.6492`
- ROC AUC: `0.6937`
- Average precision: `0.6187`
- F1: `0.5153`
- Precision: `0.6553`
- Recall: `0.4245`

Best-config seed robustness:

- Mean test balanced accuracy: `0.6586`
- Std test balanced accuracy: `0.0078`
- Min/max test balanced accuracy: `0.6492 / 0.6677`

### Uncertainty And Calibration

Participant-level bootstrap on the selected representative model:

- Test participants: 14
- Balanced accuracy point estimate: `0.6492`
- Balanced accuracy 95% CI: `[0.5436, 0.7259]`
- ROC AUC 95% CI: `[0.5567, 0.7872]`
- Average precision 95% CI: `[0.3735, 0.7773]`

Calibration:

- Brier score: `0.2126`
- Expected calibration error: `0.1256`
- Mean predicted probability: `0.4396`
- Observed positive rate: `0.3610`

Probability scores are not yet well calibrated enough to communicate as literal good-sleep probabilities. The model is more appropriate for threshold-based classification and relative ranking.

### Inference Pipeline

Reusable inference artifacts were created.

- Manifest: `data/processed/pre_sleep_forecasting/design_c_stage1/inference_package/pre_sleep_inference_manifest.json`
- Feature contract: `data/processed/pre_sleep_forecasting/design_c_stage1/inference_package/pre_sleep_inference_feature_contract.csv`
- Feature builder: `src/pre_sleep_forecasting/feature_builder.py`
- Inference script: `src/pre_sleep_forecasting/inference.py`

End-to-end smoke test succeeded:

1. Episode CSV with `participant_object_id` and `sleep_start_datetime`
2. Feature builder generated raw Stage 1 features
3. Inference script generated predictions
4. Smoke test rows: 3

### Interpretation

The selected pre-sleep MLP is the current best model for the refined real-use objective. It shows meaningful predictive signal and improved seed stability, but confidence intervals remain wide because the held-out test set contains only 14 participants.

Recommended use:

- Suitable for research-grade personal sleep-quality forecasting experiments
- Suitable for ranking or threshold-based feedback
- Not yet suitable for high-stakes medical or clinical decision-making
- Probability values should be interpreted cautiously unless calibration is improved

## Final Pre-Sleep Forecasting Package Index

The strict pre-sleep forecasting workflow has been formalized as a reusable research-grade inference package.

Primary entry points:

- Root README: README.md
- Final project status: reports/pre_sleep_forecasting_project_final_status.md
- Final artifact inventory: reports/pre_sleep_final_artifact_inventory.md
- Inference usage guide: docs/pre_sleep_inference_usage.md
- Inference package QA report: reports/pre_sleep_inference_package_qa.md
- Inference dependency list: requirements-inference.txt

Core inference files:

- Feature builder: src/pre_sleep_forecasting/feature_builder.py
- Inference runner: src/pre_sleep_forecasting/inference.py
- Manifest: data/processed/pre_sleep_forecasting/design_c_stage1/inference_package/pre_sleep_inference_manifest.json
- Feature contract: data/processed/pre_sleep_forecasting/design_c_stage1/inference_package/pre_sleep_inference_feature_contract.csv

Current final model remains:

- Experiment family: presleep_stage1_hp_tiny_dropout40_wd1e3
- Representative checkpoint: presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027
- Official threshold: 0.54
- Test balanced accuracy: 0.6492
- Participant bootstrap balanced accuracy 95% CI: [0.5436, 0.7259]

Remaining optional work is now separated from the finalized package:

- strict pre-sleep sequence-model experiments,
- calibration correction experiments,
- external or future-period validation.
